In [0]:
%sql
CREATE OR REPLACE TABLE subscription_catalog.gold_schema.data_cube_subscription_opportunities AS
SELECT 
  
  d.year,
  d.quarter,
  d.month,
  d.month_name,
  

  g.customer_country,
  g.transaction_currency,
  
  
  f.product_id,
  f.employee_id,
  ct.contract_term,
  ct.months_in_term,
  cs.close_status,
  COALESCE(cs.is_successful, 0) as is_successful,
  
  
  COUNT(DISTINCT f.opportunity_id) as total_opportunities,
  COUNT(DISTINCT f.customer_id) as unique_customers,
  SUM(f.revenue_in_gbp) as total_revenue_gbp,
  AVG(f.revenue_in_gbp) as avg_revenue_gbp,
  SUM(f.list_price) as total_list_price,
  AVG(f.contract_duration_days) as avg_contract_duration_days,
  MIN(f.start_date) as earliest_start_date,
  MAX(f.end_date) as latest_end_date,
  

  SUM(CASE WHEN COALESCE(cs.is_successful, 0) = 1 THEN 1 ELSE 0 END) as successful_opportunities,
  ROUND(SUM(CASE WHEN COALESCE(cs.is_successful, 0) = 1 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) as win_rate_pct
  
FROM subscription_catalog.gold_schema.fact_opportunities f


LEFT JOIN subscription_catalog.gold_schema.dim_date d
  ON f.start_date = d.date_value
  

LEFT JOIN subscription_catalog.gold_schema.dim_geography g
  ON f.customer_country = g.customer_country
  AND f.transaction_currency = g.transaction_currency
  

LEFT JOIN subscription_catalog.gold_schema.dim_contract_type ct
  ON f.contract_term = ct.contract_term
  

LEFT JOIN subscription_catalog.gold_schema.dim_close_status cs
  ON f.close_status = cs.close_status

GROUP BY 
  d.year,
  d.quarter,
  d.month,
  d.month_name,
  g.customer_country,
  g.transaction_currency,
  f.product_id,
  f.employee_id,
  ct.contract_term,
  ct.months_in_term,
  cs.close_status,
  COALESCE(cs.is_successful, 0)
  
ORDER BY 
  d.year DESC,
  d.quarter DESC,
  d.month DESC,
  total_revenue_gbp DESC